In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats as stats
from functools import reduce
from pathlib import Path
import os

: 

### 1. Carga de los archivos Excel
- Leemos los tres Excel y guardamos sus hojas en `excels_cargados`.
- En este bloque no filtramos, limpiamos ni unimos datos. Los siguientes bloques reutilizan esta carga.


In [ ]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Aseguramos que la carpeta data/processed exista antes de guardar el CSV
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Lista ordenada cronológicamente (importante para que keep='last' conserve el más reciente)
archivos_excel = [
    "Totalizadores Planta 2020_2022.xlsx",
    "Totalizadores Planta - 2021_2023.xlsx",
    "Totalizadores Planta 2022_2023.xlsx"
]

# 2. Cargar TODAS las hojas (sheet_name=None) en el diccionario que esperan los bloques siguientes
excels_cargados = {
    nombre: pd.read_excel(RAW_DIR / nombre, sheet_name=None)
    for nombre in archivos_excel
}

print(f"Archivos cargados correctamente: {list(excels_cargados.keys())}")


### 2. Días registrados por archivo y año
- Contamos las fechas distintas de cada archivo, sin repetirlas entre sus hojas con DIA y HORA.
- Graficamos los archivos en las filas y los años en las columnas, antes de cualquier limpieza.
- Son días registrados, no necesariamente días con 25 lecturas completas. No sumamos archivos porque comparten fechas.
- La fila «Días que difieren» cuenta fechas presentes en al menos dos archivos con datos distintos en alguna hoja/columna común; cada fecha suma una sola vez por año.
- Comparamos todas las filas del día, no solo el cierre. Ignoramos el orden de filas, pero una lectura faltante o una cantidad distinta de copias sí cuenta como diferencia. No contamos hojas/columnas exclusivas de un archivo como diferencias de valores.
- La fila «Cierre de frío distinto» compara únicamente Frio (Kw) de Consolidado EE a las 23:59, dentro de esas fechas con versiones distintas. No usa las otras features ni redondea los consumos.
- Copias del mismo valor de cierre no crean una diferencia. Si alguna versión carece de cierre válido o tiene varios valores distintos a las 23:59, el día queda «No comparable», separado del conteo. Detalle disponible en `comparacion_cierres_frio`.
- «Lecturas distintas antes del cierre»: comparamos las filas excluyendo las 23:59. Cuenta cambios en valores de cualquier feature común, celdas faltantes, horarios o cantidad de filas; aunque ambos archivos tengan los mismos horarios, valores diferentes sí cuentan. Ignoramos el orden de filas y contamos cada fecha una vez.
- «Diferencias en sus cierres»: comparamos todas las features comunes de las filas de las 23:59, aunque también cambien otras horas. Incluye cambios de valores, faltantes o cantidad de filas de cierre. No exige que el resto del día sea idéntico.
- «Cierres solo por datos faltantes»: miramos únicamente las filas de las 23:59. Debe existir alguna celda presente en una versión y nula en otra, sin ninguna contradicción entre valores presentes en ambas, en las hojas/columnas comunes. Las diferencias de otras horas no influyen. Un cierre ausente o distinta cantidad de filas de cierre no se clasifica como solo celdas faltantes.
- Las filas pueden superponerse: los cambios en el cierre se cuentan independientemente de los cambios de valores, horarios o faltantes del resto del día. Cada fecha cuenta una vez por fila; no deben sumarse las filas como si fueran categorías excluyentes.
- Ejemplo que cuenta en la última fila: [20, 40, 50] frente a [null, 40, null]. Si el 40 cambia a 41, ya no cuenta. Cada fecha cuenta una vez y se exige compatibilidad entre todas sus versiones comparadas.


In [ ]:
fechas_por_archivo = {}
for nombre_archivo, hojas in excels_cargados.items():
    fechas_archivo = set()
    for df in hojas.values():
        if 'DIA' in df.columns and 'HORA' in df.columns:
            fechas_archivo.update(pd.to_datetime(df['DIA'], errors='coerce').dropna().dt.normalize())
    fechas_por_archivo[nombre_archivo] = pd.DatetimeIndex(sorted(fechas_archivo))

conteos = {nombre: pd.Series(fechas.year).value_counts() for nombre, fechas in fechas_por_archivo.items()}
dias_por_archivo_anio = pd.DataFrame.from_dict(conteos, orient='index').reindex(archivos_excel).fillna(0).astype(int).sort_index(axis=1)
# Comparamos hojas y columnas comunes; un cambio de esquema por sí solo no cuenta.
def firmas_por_dia(izquierda, derecha):
    columnas = sorted(set(izquierda.columns) & set(derecha.columns), key=str)
    juntas = pd.concat([izquierda[columnas], derecha[columnas]], ignore_index=True)
    fechas = pd.to_datetime(juntas['DIA'], errors='coerce').dt.normalize()
    juntas['HORA'] = juntas['HORA'].astype(str).str.strip()
    # Un dtype común evita diferencias artificiales entre, por ejemplo, 1 y 1.0.
    valores = juntas.drop(columns='DIA')
    hashes = pd.util.hash_pandas_object(valores, index=False)
    corte = len(izquierda)
    firmas = []
    for tramo in [slice(0, corte), slice(corte, None)]:
        tabla = pd.DataFrame({'Fecha': fechas.iloc[tramo], 'Firma': hashes.iloc[tramo]})
        firmas.append(tabla.groupby('Fecha')['Firma'].agg(lambda x: tuple(sorted(x))).to_dict())
    return firmas

from scipy.optimize import linear_sum_assignment

def comparar_faltantes_cierre(a, b, columnas):
    if len(a) != len(b):
        return False, True  # Un cierre ausente o filas extra no son solo celdas faltantes.
    if a.empty:
        return False, False
    x, y = a[columnas].to_numpy(dtype=object), b[columnas].to_numpy(dtype=object)
    mx, my = pd.isna(x), pd.isna(y)
    x[mx], y[my] = 0, 0
    contradicciones = ((x[:, None, :] != y[None, :, :]) & ~mx[:, None, :] & ~my[None, :, :]).sum(axis=2)
    faltantes = (mx[:, None, :] != my[None, :, :]).sum(axis=2)
    peso = int(faltantes.max(initial=0) * len(x) + 1)
    filas, columnas_asignadas = linear_sum_assignment(contradicciones * peso + faltantes)
    return bool(faltantes[filas, columnas_asignadas].sum()), bool(contradicciones[filas, columnas_asignadas].sum())

def clasificar_diferencia_dia(a, b):
    cierre_a = a.loc[a['HORA'].astype(str).str.strip().isin(['23:59', '23:59:00'])]
    cierre_b = b.loc[b['HORA'].astype(str).str.strip().isin(['23:59', '23:59:00'])]
    firma_cierre_a, firma_cierre_b = firmas_por_dia(cierre_a, cierre_b)
    cierre_diferente = firma_cierre_a != firma_cierre_b
    # Comparamos todas las columnas comunes de las lecturas anteriores al cierre.
    previas_a = a.loc[~a['HORA'].astype(str).str.strip().isin(['23:59', '23:59:00'])]
    previas_b = b.loc[~b['HORA'].astype(str).str.strip().isin(['23:59', '23:59:00'])]
    firma_previas_a, firma_previas_b = firmas_por_dia(previas_a, previas_b)
    lecturas_previas_distintas = firma_previas_a != firma_previas_b
    columnas = sorted((set(a.columns) & set(b.columns)) - {'DIA', 'HORA'}, key=str)
    cierre_faltantes, cierre_incompatible = comparar_faltantes_cierre(cierre_a, cierre_b, columnas)
    grupos_a = {str(h).strip(): g for h, g in a.groupby(a['HORA'].astype(str).str.strip())}
    grupos_b = {str(h).strip(): g for h, g in b.groupby(b['HORA'].astype(str).str.strip())}
    horarios_distintos = {h: len(g) for h, g in grupos_a.items()} != {h: len(g) for h, g in grupos_b.items()}
    if horarios_distintos:
        return {'horarios': True, 'conflicto': False, 'fuera_cierre': True, 'cierre': cierre_diferente, 'cierre_faltantes': cierre_faltantes, 'cierre_incompatible': cierre_incompatible, 'lecturas_previas': lecturas_previas_distintas}
    conflicto, fuera_cierre = False, False
    for hora in grupos_a:
        x = grupos_a[hora][columnas].to_numpy(dtype=object)
        y = grupos_b[hora][columnas].to_numpy(dtype=object)
        mx, my = pd.isna(x), pd.isna(y)
        x[mx], y[my] = 0, 0
        presentes = ~mx[:, None, :] & ~my[None, :, :]
        contradicciones = ((x[:, None, :] != y[None, :, :]) & presentes).sum(axis=2)
        cambios = contradicciones + (mx[:, None, :] != my[None, :, :]).sum(axis=2)
        # Emparejamos las filas de la misma hora sin depender de su orden.
        peso = int(cambios.max(initial=0) * len(x) + 1)
        filas, columnas_asignadas = linear_sum_assignment(contradicciones * peso + cambios)
        conflicto |= bool(contradicciones[filas, columnas_asignadas].sum())
        if hora not in ['23:59', '23:59:00']:
            fuera_cierre |= bool(cambios[filas, columnas_asignadas].sum())
    return {'horarios': False, 'conflicto': conflicto, 'fuera_cierre': fuera_cierre, 'cierre': cierre_diferente, 'cierre_faltantes': cierre_faltantes, 'cierre_incompatible': cierre_incompatible, 'lecturas_previas': lecturas_previas_distintas}

clasificacion_por_fecha = {}

dias_versiones_distintas = set()
for i, archivo_a in enumerate(archivos_excel):
    for archivo_b in archivos_excel[i + 1:]:
        fechas_compartidas = set(fechas_por_archivo[archivo_a]) & set(fechas_por_archivo[archivo_b])
        if not fechas_compartidas:
            continue
        hojas_a, hojas_b = excels_cargados[archivo_a], excels_cargados[archivo_b]
        for hoja in sorted(set(hojas_a) & set(hojas_b)):
            a, b = hojas_a[hoja], hojas_b[hoja]
            if not {'DIA', 'HORA'}.issubset(a.columns) or not {'DIA', 'HORA'}.issubset(b.columns):
                continue
            firmas_a, firmas_b = firmas_por_dia(a, b)
            fechas_diferentes = {fecha for fecha in fechas_compartidas if firmas_a.get(fecha, ()) != firmas_b.get(fecha, ())}
            dias_versiones_distintas.update(fechas_diferentes)
            if fechas_diferentes:
                grupos_a = dict(tuple(a.groupby(pd.to_datetime(a['DIA'], errors='coerce').dt.normalize())))
                grupos_b = dict(tuple(b.groupby(pd.to_datetime(b['DIA'], errors='coerce').dt.normalize())))
                for fecha in fechas_diferentes:
                    resultado = clasificar_diferencia_dia(grupos_a.get(fecha, a.iloc[:0]), grupos_b.get(fecha, b.iloc[:0]))
                    acumulado = clasificacion_por_fecha.setdefault(fecha, {'horarios': False, 'conflicto': False, 'fuera_cierre': False, 'cierre': False, 'cierre_faltantes': False, 'cierre_incompatible': False, 'lecturas_previas': False})
                    for clave in acumulado:
                        acumulado[clave] |= resultado[clave]

# Comparamos solo cierres de frío en los días con versiones distintas ya detectados.
cierres_por_archivo = {}
for archivo, hojas in excels_cargados.items():
    if 'Consolidado EE' not in hojas or not {'DIA', 'HORA', 'Frio (Kw)'}.issubset(hojas['Consolidado EE'].columns):
        raise ValueError(f"Falta DIA, HORA o Frio (Kw) en Consolidado EE de {archivo}")
    ee = hojas['Consolidado EE']
    fechas = pd.to_datetime(ee['DIA'], errors='coerce').dt.normalize()
    es_cierre = ee['HORA'].astype(str).str.strip().isin(['23:59:00', '23:59'])
    cierres = pd.DataFrame({'Fecha': fechas[es_cierre], 'Frio': pd.to_numeric(ee.loc[es_cierre, 'Frio (Kw)'], errors='coerce')})
    cierres_por_archivo[archivo] = {fecha: grupo['Frio'].to_numpy() for fecha, grupo in cierres.groupby('Fecha')}

comparaciones_cierre = []
for fecha in sorted(dias_versiones_distintas):
    versiones = [archivo for archivo in archivos_excel if fecha in fechas_por_archivo[archivo]]
    valores_cierre, motivos = {}, []
    for archivo in versiones:
        lecturas = cierres_por_archivo[archivo].get(fecha, np.array([]))
        distintos = np.unique(lecturas[np.isfinite(lecturas)])
        valores_cierre[archivo] = distintos.tolist()
        if len(lecturas) == 0:
            motivos.append(f'{archivo}: falta 23:59')
        elif not np.isfinite(lecturas).all():
            motivos.append(f'{archivo}: frío faltante o inválido')
        elif len(distintos) != 1:
            motivos.append(f'{archivo}: varios cierres contradictorios')
    if motivos or len(versiones) < 2:
        estado = 'No comparable'
    else:
        estado = 'Distinto' if len({v[0] for v in valores_cierre.values()}) > 1 else 'Igual'
    comparaciones_cierre.append({'Fecha': fecha, 'Estado': estado, 'Cierres por archivo': valores_cierre, 'Motivo': '; '.join(motivos)})
comparacion_cierres_frio = pd.DataFrame(comparaciones_cierre, columns=['Fecha', 'Estado', 'Cierres por archivo', 'Motivo'])
dias_cierre_distinto = {fila['Fecha'] for fila in comparaciones_cierre if fila['Estado'] == 'Distinto'}
dias_cierre_no_comparable = {fila['Fecha'] for fila in comparaciones_cierre if fila['Estado'] == 'No comparable'}
cierres_distintos_por_anio = pd.Series([fecha.year for fecha in dias_cierre_distinto], dtype='int64').value_counts().reindex(dias_por_archivo_anio.columns, fill_value=0)
assert dias_cierre_distinto.issubset(dias_versiones_distintas)

dias_lecturas_previas_distintas = {f for f, r in clasificacion_por_fecha.items() if r['lecturas_previas']}
dias_cierres_solo_faltantes = {f for f, r in clasificacion_por_fecha.items() if r['cierre_faltantes'] and not r['cierre_incompatible']}
dias_cierres_diferentes = {f for f, r in clasificacion_por_fecha.items() if r['cierre']}
# El cierre se evalúa independientemente de los cambios durante otras horas.
def contar_por_anio(fechas):
    return pd.Series([f.year for f in fechas], dtype='int64').value_counts().reindex(dias_por_archivo_anio.columns, fill_value=0)
assert dias_cierres_diferentes.issubset(dias_versiones_distintas)
assert dias_cierres_solo_faltantes.issubset(dias_cierres_diferentes)

dias_distintos_por_anio = pd.Series([fecha.year for fecha in dias_versiones_distintas], dtype='int64').value_counts().reindex(dias_por_archivo_anio.columns, fill_value=0)
cobertura_con_diferencias = dias_por_archivo_anio.copy()
cobertura_con_diferencias.loc['Días que difieren de cada año'] = dias_distintos_por_anio
cobertura_con_diferencias.loc['Cantidad de días con cierre de frío distinto'] = cierres_distintos_por_anio
cobertura_con_diferencias.loc['Días con lecturas distintas antes del cierre'] = contar_por_anio(dias_lecturas_previas_distintas)
cobertura_con_diferencias.loc['Días con diferencias en sus cierres (23:59)'] = contar_por_anio(dias_cierres_diferentes)
cobertura_con_diferencias.loc['Días con diferencias en cierres solo por datos faltantes'] = contar_por_anio(dias_cierres_solo_faltantes)
valores = cobertura_con_diferencias.to_numpy()
etiquetas_archivos = [nombre.replace('Totalizadores Planta', 'Excel').replace(' - ', ' ').replace('.xlsx', '').replace('_', '–') for nombre in dias_por_archivo_anio.index]
etiquetas_archivos.extend(['Días que difieren\nde cada año', 'Días con cierre\nde frío distinto', 'Días con lecturas distintas\nantes del cierre', 'Días con diferencias en\nsus cierres (23:59)', 'Días con diferencias en cierres\nsolo por datos faltantes'])
fig, ax = plt.subplots(figsize=(12, 8.5))
fig.patch.set_facecolor('#fafcff')
mascara_ultima_fila = np.zeros(valores.shape, dtype=bool)
mascara_ultima_fila[len(archivos_excel):, :] = True
ax.imshow(np.ma.masked_where(mascara_ultima_fila, valores), cmap='Blues', vmin=0, vmax=366, aspect='auto')
ax.imshow(np.ma.masked_where(~mascara_ultima_fila, valores), cmap='Reds', vmin=0, vmax=366, aspect='auto')
ax.set_xticks(range(len(dias_por_archivo_anio.columns)), dias_por_archivo_anio.columns, fontsize=12)
ax.xaxis.tick_top()
ax.set_yticks(range(len(etiquetas_archivos)), etiquetas_archivos, fontsize=12)
for i in range(valores.shape[0]):
    for j in range(valores.shape[1]):
        ax.text(j, i, f'{valores[i, j]} días' if valores[i, j] or i >= len(archivos_excel) else '—', ha='center', va='center', fontsize=15, fontweight='bold', color='white' if valores[i, j] > 200 else '#18324a')
ax.tick_params(length=0, pad=10)
for borde in ax.spines.values():
    borde.set_visible(False)
ax.set_xticks(np.arange(-.5, valores.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, valores.shape[0], 1), minor=True)
ax.grid(which='minor', color='white', linewidth=4)
ax.tick_params(which='minor', length=0)
ax.axhline(len(archivos_excel) - .5, color='#18324a', linewidth=2)
fig.suptitle('Días registrados y versiones diferentes', fontsize=17, fontweight='bold', x=.57, y=.98, color='#18324a')
fig.text(.02, .025, 'Comparación de hojas/columnas comunes. Cada fecha cuenta una vez por fila.\nLas filas pueden superponerse: un día puede tener diferencias en lecturas previas y también en su cierre.', fontsize=10, color='#526171')
plt.tight_layout(rect=[0, .08, 1, .90])
plt.show()
if dias_cierre_no_comparable:
    no_comparables_por_anio = pd.Series([fecha.year for fecha in dias_cierre_no_comparable]).value_counts().sort_index()
    print('Cierres de frío no comparables (excluidos de su conteo): ' + ', '.join(f'{anio}: {cantidad} días' for anio, cantidad in no_comparables_por_anio.items()) + '. Detalle en comparacion_cierres_frio.')


### 3. Diagnóstico de días con más de 25 filas
- Revisamos los días excedidos de cada hoja/archivo sin modificar los datos cargados.
- Primer gráfico: todos los valores de HORA repetidos dentro del mismo día, hoja y archivo, coincidan o no las demás columnas.
- Segundo gráfico: todos los valores de HORA fuera de los 25 horarios exactos esperados (00:00 a 23:00 y 23:59), incluyendo textos o vacíos si existen. Un desfase de segundos no implica por sí solo un error.
- Las barras cuentan días distintos: una fecha suma una vez por valor de hora aunque aparezca en varias hojas/archivos. Los valores repetidos y anómalos pueden coincidir.
- Guardamos los originales de los casos en `jornadas_detectadas` y su detalle en `detalle_jornadas`. El tratamiento se realiza después.


In [ ]:
import re
from IPython.display import display

horas_normales = {f'{h:02d}:00:00' for h in range(24)} | {'23:59:00'}
registros_revision = []
horas_repetidas_por_dia, horas_anomalas_por_dia = set(), set()
jornadas_detectadas, fechas_registradas = {}, set()

def normalizar_hora(valor):
    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    coincidencia = re.fullmatch(r'(\d{1,2}):(\d{2})(?::(\d{2}))?', texto)
    if not coincidencia:
        return None
    h, m, s = (int(v or 0) for v in coincidencia.groups())
    return f'{h:02d}:{m:02d}:{s:02d}' if h < 24 and m < 60 and s < 60 else None

def filas_sin_mediciones(dia):
    columnas = [c for c in dia.columns if c not in ['DIA', 'HORA']]
    return dia[columnas].apply(lambda c: c.isna() | c.map(lambda v: isinstance(v, str) and not v.strip())).all(axis=1) if columnas else pd.Series(False, index=dia.index)

for archivo, hojas in excels_cargados.items():
    for hoja, df in hojas.items():
        if 'DIA' not in df.columns or 'HORA' not in df.columns:
            continue
        fechas = pd.to_datetime(df['DIA'], errors='coerce').dt.normalize()
        fechas_registradas.update(fechas.dropna().unique())
        conteo = fechas.value_counts()
        for fecha, dia in df.loc[fechas.isin(conteo[conteo > 25].index)].groupby(fechas):
            jornadas_detectadas[(archivo, hoja, fecha)] = dia.copy()
            duplicadas = dia.duplicated(keep='first')
            vacias = filas_sin_mediciones(dia)
            horas = dia['HORA'].map(normalizar_hora)
            anomalas = ~horas.isin(horas_normales)
            etiquetas_hora = dia['HORA'].map(lambda v: '(vacío)' if pd.isna(v) or not str(v).strip() else str(v).strip())
            # Cuenta repetición de HORA, aunque las otras columnas sean idénticas o diferentes.
            claves_hora = horas.where(horas.notna(), etiquetas_hora)
            for valor in claves_hora[claves_hora.duplicated(keep=False)].unique():
                horas_repetidas_por_dia.add((fecha, valor))
            for valor in etiquetas_hora[anomalas].unique():
                horas_anomalas_por_dia.add((fecha, valor))
            registros_revision.append({'Archivo': archivo, 'Hoja': hoja, 'Fecha': fecha, 'Filas originales': len(dia), 'Copias sobrantes': int(duplicadas.sum()), 'Grupos duplicados': dia[dia.duplicated(keep=False)].drop_duplicates().shape[0], 'Filas sin mediciones': int(vacias.sum()), 'Filas con hora anómala': int(anomalas.sum())})

detalle_jornadas = pd.DataFrame(registros_revision)
dias_con_exceso = detalle_jornadas['Fecha'].nunique() if not detalle_jornadas.empty else 0
print(f'Días registrados: {len(fechas_registradas)} | Días con más de 25 filas: {dias_con_exceso}.')

def contar_dias_por_hora(pares):
    tabla = pd.DataFrame(sorted(pares), columns=['Fecha', 'Hora'])
    return tabla.groupby('Hora')['Fecha'].nunique().sort_index()

conteo_horas_repetidas = contar_dias_por_hora(horas_repetidas_por_dia)
conteo_horas_anomalas = contar_dias_por_hora(horas_anomalas_por_dia)
altura = max(4, 0.30 * max(len(conteo_horas_repetidas), len(conteo_horas_anomalas)) + 1.8)
fig, ejes = plt.subplots(1, 2, figsize=(12, altura))
for ax, conteos, titulo, color in [(ejes[0], conteo_horas_repetidas, 'Horas que se repiten', '#2878ad'), (ejes[1], conteo_horas_anomalas, 'Valores fuera de los horarios esperados', '#d9534f')]:
    if conteos.empty:
        ax.text(.5, .5, 'No se detectaron casos', ha='center', va='center', transform=ax.transAxes)
        ax.set_yticks([])
    else:
        barras = ax.barh(conteos.index, conteos.values, color=color, height=.7)
        ax.invert_yaxis()
        ax.bar_label(barras, padding=4, fontsize=10)
        ax.set_xlim(0, max(conteos.max() * 1.15, 1))
        ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Cantidad de días distintos')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=.15)
    ax.set_axisbelow(True)
fig.suptitle('Diagnóstico de días con más de 25 filas', fontsize=15, fontweight='bold')
fig.text(.5, .015, 'Cada fecha cuenta una vez por valor de hora. Repetición dentro de una misma hoja/archivo.\nFuera del horario esperado no significa necesariamente inválido; no se corrigen valores aquí.', ha='center', fontsize=10)
plt.tight_layout(rect=[0, .07, 1, .95])
plt.show()


### 4. Tratamiento de los casos detectados
- Trabajamos con copias: los Excel cargados y el diagnóstico permanecen intactos.
- Quitamos duplicados exactos conservando una copia; quitamos filas sin mediciones solo si quedan exactamente los 25 horarios esperados.
- No corregimos automáticamente horas anómalas ni elegimos entre valores contradictorios.
- Si aún quedan más de 25 filas, guardamos el caso original en `jornadas_para_revisar` y lo excluimos de la unión por ahora. Los demás pasan a `datos_por_hoja`; tener hasta 25 filas no certifica su calidad.
- Este bloque prepara los datos; la unión ocurre en el siguiente.
- Conservamos también las tablas tratadas por archivo para comparar versiones antes de unirlas.


In [ ]:
datos_por_hoja, jornadas_para_revisar, registro_tratamiento = {}, {}, []
datos_tratados_por_archivo = {}
for archivo, hojas in excels_cargados.items():
    for hoja, df in hojas.items():
        if 'DIA' not in df.columns or 'HORA' not in df.columns:
            continue
        fechas = pd.to_datetime(df['DIA'], errors='coerce').dt.normalize()
        fechas_exceso = [fecha for a, h, fecha in jornadas_detectadas if a == archivo and h == hoja]
        partes = [df.loc[~fechas.isin(fechas_exceso)].copy()]
        for fecha in fechas_exceso:
            original = jornadas_detectadas[(archivo, hoja, fecha)]
            limpio = original.drop_duplicates(keep='first').copy()
            sin_vacias = limpio.loc[~filas_sin_mediciones(limpio)]
            if len(sin_vacias) == 25 and set(sin_vacias['HORA'].map(normalizar_hora)) == horas_normales:
                limpio = sin_vacias.copy()
            conservar = len(limpio) <= 25
            if conservar:
                partes.append(limpio)
            else:
                jornadas_para_revisar[(archivo, hoja, fecha)] = original.copy()
            registro_tratamiento.append({'Archivo': archivo, 'Hoja': hoja, 'Fecha': fecha, 'Filas originales': len(original), 'Filas tras tratamiento': len(limpio), 'Estado': 'Conservar' if conservar else 'Revisar aparte'})
        preparado = pd.concat(partes).sort_index()
        preparado['DIA'] = pd.to_datetime(preparado['DIA'], errors='coerce').dt.normalize()
        preparado['HORA'] = preparado['HORA'].astype(str).str.strip()
        if not preparado.empty:
            datos_por_hoja.setdefault(hoja, []).append(preparado)
            datos_tratados_por_archivo.setdefault(archivo, {})[hoja] = preparado.copy()
detalle_tratamiento = pd.DataFrame(registro_tratamiento)
conservados = int((detalle_tratamiento['Estado'] == 'Conservar').sum()) if not detalle_tratamiento.empty else 0
print(f'Casos por archivo y hoja: {conservados} recuperados | {len(jornadas_para_revisar)} pendientes de revisión. Originales conservados.')


### 4.1. Features que más cambian entre versiones, después del tratamiento
- Comparamos todas las horas coincidentes de cada fecha entre archivos, en hojas y columnas comunes; no solo el cierre.
- Ordenamos las features por cantidad de días distintos con diferencias. La misma fecha cuenta una vez por feature, aunque tenga varias horas o pares de archivos diferentes.
- Rojo: valores presentes en ambas versiones que no coinciden. Amarillo: diferencias solo por faltantes en esa feature; si el día tiene ambos problemas, cuenta en rojo.
- Mostramos todas las features con diferencias en paneles de hasta 20, de mayor a menor. Las que coinciden siempre no aparecen. El detalle completo queda en `ranking_features`, incluido el número de días comparables.
- Usamos únicamente los casos conservados por el punto 4. Los días apartados, las horas sin contraparte y las claves que aún tienen varias filas quedan fuera de esta comparación: no representan diferencias comprobadas de una feature alineada.
- Este ranking identifica dónde cambian los datos, no determina cuál versión es correcta ni qué variables predicen mejor el frío. Incluye metas e identificadores si difieren.


In [ ]:
# Conservamos la procedencia antes de que la unión elija una versión.
from collections import defaultdict

dias_valores_distintos = defaultdict(set)
dias_faltantes_distintos = defaultdict(set)
dias_comparables_feature = defaultdict(set)
tablas_comparables, claves_ambiguas = {}, set()
for archivo, hojas in datos_tratados_por_archivo.items():
    tablas_comparables[archivo] = {}
    for hoja, df in hojas.items():
        tabla = df.loc[df['DIA'].notna()].copy()
        repetidas = tabla.duplicated(['DIA', 'HORA'], keep=False)
        for fecha, hora in tabla.loc[repetidas, ['DIA', 'HORA']].itertuples(index=False, name=None):
            claves_ambiguas.add((archivo, hoja, fecha, hora))
        # No elegimos arbitrariamente una fila cuando todavía quedan varias por hora.
        tablas_comparables[archivo][hoja] = tabla.loc[~repetidas].set_index(['DIA', 'HORA'])

for i, archivo_a in enumerate(archivos_excel):
    for archivo_b in archivos_excel[i + 1:]:
        hojas_a = tablas_comparables.get(archivo_a, {})
        hojas_b = tablas_comparables.get(archivo_b, {})
        for hoja in sorted(set(hojas_a) & set(hojas_b)):
            a, b = hojas_a[hoja], hojas_b[hoja]
            claves = a.index.intersection(b.index)
            if claves.empty:
                continue
            fechas = claves.get_level_values('DIA')
            for columna in sorted(set(a.columns) & set(b.columns), key=str):
                feature = f'{columna} ({hoja})'
                x, y = a.loc[claves, columna], b.loc[claves, columna]
                x = x.mask(x.map(lambda v: isinstance(v, str) and not v.strip()))
                y = y.mask(y.map(lambda v: isinstance(v, str) and not v.strip()))
                presentes = x.notna() & y.notna()
                distintos = presentes & ~x.eq(y).fillna(False)
                faltantes = x.isna() ^ y.isna()
                dias_comparables_feature[feature].update(fechas)
                dias_valores_distintos[feature].update(fechas[distintos.to_numpy()])
                dias_faltantes_distintos[feature].update(fechas[faltantes.to_numpy()])

registros_ranking = []
for feature in dias_comparables_feature:
    valores = dias_valores_distintos[feature]
    # Si ese día tiene ambos problemas, lo contamos solo en valores distintos.
    solo_faltantes = dias_faltantes_distintos[feature] - valores
    if valores or solo_faltantes:
        registros_ranking.append({'Feature': feature, 'Días con valores distintos': len(valores), 'Días solo con faltantes': len(solo_faltantes), 'Días con diferencias': len(valores | solo_faltantes), 'Días comparables': len(dias_comparables_feature[feature])})
ranking_features = pd.DataFrame(registros_ranking, columns=['Feature', 'Días con valores distintos', 'Días solo con faltantes', 'Días con diferencias', 'Días comparables'])
ranking_features = ranking_features.sort_values(['Días con diferencias', 'Feature'], ascending=[False, True]).reset_index(drop=True)

if ranking_features.empty:
    print('No hay diferencias de features en las lecturas comparables después del tratamiento.')
else:
    print(f'{len(ranking_features)} features con diferencias. Días únicos por feature; comparación de todas las horas coincidentes.')
    # Mostramos el ranking completo en paneles legibles, sin incluir features idénticas.
    por_panel = 20
    paneles = (len(ranking_features) + por_panel - 1) // por_panel
    for panel in range(paneles):
        inicio = panel * por_panel
        datos = ranking_features.iloc[inicio:inicio + por_panel]
        fig, ax = plt.subplots(figsize=(12, max(3, .32 * len(datos) + 1.5)))
        posiciones = np.arange(len(datos))
        valores = datos['Días con valores distintos'].to_numpy()
        faltantes = datos['Días solo con faltantes'].to_numpy()
        ax.barh(posiciones, valores, color='#d9534f', label='Valores distintos')
        ax.barh(posiciones, faltantes, left=valores, color='#e9ad45', label='Solo faltantes')
        etiquetas = [f'{inicio + j + 1}. {feature}' for j, feature in enumerate(datos['Feature'])]
        ax.set_yticks(posiciones, etiquetas, fontsize=9)
        ax.invert_yaxis()
        for j, total in enumerate(datos['Días con diferencias']):
            ax.text(total + .4, j, str(total), va='center', fontsize=9)
        ax.set_xlim(0, ranking_features['Días con diferencias'].max() * 1.1 + 1)
        ax.xaxis.get_major_locator().set_params(integer=True)
        ax.set_xlabel('Cantidad de días distintos con discrepancias')
        ax.set_title(f'Features que cambian entre versiones · puestos {inicio + 1}–{inicio + len(datos)}', fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='x', alpha=.15)
        ax.set_axisbelow(True)
        ax.legend(loc='upper center', bbox_to_anchor=(.5, -.13), ncol=2, frameon=False)
        plt.tight_layout()
        plt.show()
if claves_ambiguas:
    print(f'Se excluyeron {len(claves_ambiguas)} combinaciones archivo/hoja/fecha/hora con varias filas pendientes de resolver.')


### 5. Unión de archivos y hojas
- Unimos los archivos de cada hoja y resolvemos solapamientos de DIA/HORA conservando el último archivo de la lista.
- Agregamos el nombre de la hoja a las columnas para identificar su origen.
- Cruzamos todas las hojas, conservamos las fechas/horas disponibles y ordenamos la tabla final.
- Mostramos la cantidad final de filas y columnas.


In [ ]:
dfs_listos_para_cruzar = []
print("\nUnificando hojas y eliminando solapamientos...")

for nombre_hoja, lista_dfs in datos_por_hoja.items():
    # Juntamos todos los dataframes de la misma hoja a lo largo de los distintos archivos
    df_hoja_completa = pd.concat(lista_dfs, ignore_index=True)
    
    # ELIMINAR SOLAPAMIENTO: Mantenemos el último (el del archivo más reciente)
    df_hoja_completa = df_hoja_completa.drop_duplicates(subset=['DIA', 'HORA'], keep='last')
    
    # Renombrar columnas agregando el nombre de la hoja (excepto DIA y HORA)
    columnas_valores = [col for col in df_hoja_completa.columns if col not in ['DIA', 'HORA']]
    renombres = {col: f"{col} ({nombre_hoja})" for col in columnas_valores}
    df_hoja_completa.rename(columns=renombres, inplace=True)
    
    dfs_listos_para_cruzar.append(df_hoja_completa)


if dfs_listos_para_cruzar:
    # Hacemos un merge conservando DIA y HORA
    df_final = reduce(lambda left, right: pd.merge(left, right, on=['DIA', 'HORA'], how='outer'), dfs_listos_para_cruzar)
    
    # Ordenamos de forma cronológica por día y hora
    df_final = df_final.sort_values(['DIA', 'HORA']).reset_index(drop=True)
    
    print(f"Listo: {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")

EL criterio de desempate fue tomar el excel más reciente

Análisis Descriptivo y Versionado Inicial de Datos

ACA tendria que implementar el sistema de checksums

Convertimos las comas en puntos y las columnas a flotantes

In [ ]:
df_final.head()

In [ ]:
# 1. Identificar las columnas correctas explícitamente
columnas_a_float = [col for col in df_final.columns if col not in ['DIA', 'HORA']]

# 2. Reemplazar comas por puntos y convertir a numérico columna por columna
for col in columnas_a_float:
    # A. Nos aseguramos de tratar la columna como texto, y cambiamos ',' por '.'
    # Usamos regex=False porque es más rápido y exacto para buscar un simple caracter
    df_final[col] = df_final[col].astype(str).str.replace(',', '.', regex=False)
    
    # B. Forzamos la conversión a float. Si dice "N/A" o algo raro, se vuelve NaN
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# Opcional: Verificar que funcionó
print("Tipos de datos actualizados:")
print(df_final[columnas_a_float].dtypes.head(3)) # Muestra los tipos de las primeras 3 columnas

In [ ]:
ruta_csv_completo = PROCESSED_DIR / "dataset_unificado_completo.csv"
df_final.to_csv(ruta_csv_completo, index=False)

print(f"Dataset completo guardado en: {ruta_csv_completo} ({df_final.shape[0]} filas, {df_final.shape[1]} columnas)")

In [ ]:
df_final.info(verbose=True, show_counts=True)

Miramos los nulos de las columnas

In [ ]:
nulos_por_columna = df_final.isna().sum()
# print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False))
print(nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False).to_string())

Ver rango de valores

In [ ]:
columnas_numericas = [col for col in df_final.columns if col not in ['DIA', 'HORA']]
print("Rango de valores por columna:")
for col in columnas_numericas:
    minimo = df_final[col].min()
    maximo = df_final[col].max()
    print(f"- {col}: {minimo}  --->  {maximo}")

Detección de Outliers, que hago con los totalizadores??? los corrijo antes?

Miramos los de las 23:59

In [ ]:
df = df_final[df_final['HORA'].astype(str).str.startswith('23:59')].copy()

In [ ]:
def detectar_outliers_iqr(dataframe):
    # Seleccionar solo las columnas numéricas para evitar errores
    cols_numericas = dataframe.select_dtypes(include=['float64', 'int64']).columns
    
    resumen_outliers = {}
    
    for col in cols_numericas:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        
        # Contar cuántos registros superan los límites
        cantidad = ((dataframe[col] < lim_inf) | (dataframe[col] > lim_sup)).sum()
        
        if cantidad > 0:
            resumen_outliers[col] = cantidad
            
    return pd.Series(resumen_outliers).sort_values(ascending=False)

# Ejecutar la función
print(detectar_outliers_iqr(df))

Buscamos los dias con menos de 25 registros

In [ ]:
conteo_por_dia = df_final.groupby('DIA').size()

# 2. Filtrar para quedarnos solo con los días que tienen menos de 24 lecturas
dias_incompletos = conteo_por_dia[conteo_por_dia < 25]

# 3. Mostrar el resultado
print("Días con menos de 24 lecturas horarias:")
if dias_incompletos.empty:
    print("¡Excelente! No hay días con menos de 25 lecturas.")
else:
    # Mostramos la fecha y la cantidad de lecturas que tiene
    for fecha, cantidad in dias_incompletos.items():
        print(f" - {fecha.strftime('%Y-%m-%d')}: tiene {cantidad} lecturas")

Buscamos los dias en que los totalizadores no se reiniciaron

In [ ]:
df_medianoche = df_final[df_final['HORA'].astype(str).str.startswith('00:00')].copy()

# 3. Diccionario para guardar el reporte de fallas
# Formato: { 'Fecha': ['Totalizador1: valor', 'Totalizador2: valor'] }
reporte_fallas = {}

# 4. Iterar sobre las filas de la medianoche para buscar valores mayores a 0
for index, row in df_medianoche.iterrows():
    fecha_str = row['DIA'].strftime('%Y-%m-%d')
    fallas_del_dia = []
    
    for col in columnas_numericas:
        valor = row[col]
        # Verificamos que no sea NaN (nulo) y que sea mayor estricto que 0
        if pd.notna(valor) and valor > 0:
            fallas_del_dia.append(f"{col}: {valor}")
            
    # Si encontramos al menos un totalizador que no se reinició ese día, lo guardamos
    if fallas_del_dia:
        reporte_fallas[fecha_str] = fallas_del_dia

# 5. Imprimir el reporte de forma legible
print("=== REPORTE DE TOTALIZADORES QUE NO SE REINICIARON A LAS 00:00 ===\n")

if not reporte_fallas:
    print("¡Excelente! Todos los totalizadores se reiniciaron a 0 correctamente en la medianoche.")
else:
    for fecha, lista_fallas in reporte_fallas.items():
        print(f"Fecha: {fecha}")
        for falla in lista_fallas:
            print(f"  - {falla}")
        print("-" * 40)

Análisis temporal

Creamos la columna objetivo

In [ ]:
df = df.sort_values('DIA').reset_index(drop=True)
df['Target_Frio_Manana'] = df['Frio (Kw) (Consolidado EE)'].shift(-1)

In [ ]:


plt.figure(figsize=(15, 6))

# Generar el gráfico de líneas
plt.plot(df['DIA'], df['Frio (Kw) (Consolidado EE)'], 
         color='#1f77b4', linewidth=1.5, label='Frio (Kw) Acumulado Diario')

# Configuración de etiquetas y título
plt.title('Consumo Diario de Frío (2020 - 2023)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Consumo de Frío (Kw)', fontsize=12)

# --- NUEVO CÓDIGO PARA EL EJE X ---
ax = plt.gca() # Obtener los ejes actuales

# Mostrar las etiquetas cada 6 meses
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

# Darle formato a la fecha (por ejemplo: '2020-01' para Año-Mes. Puedes usar '%Y-%m-%d' para día también)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Rotar las fechas 45 grados para que no se superpongan
plt.xticks(rotation=45)
# ----------------------------------

# Formato visual
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Ajustar los márgenes automáticamente para que no se corten las etiquetas
plt.tight_layout()

# Mostrar el gráfico en pantalla
plt.show()

In [ ]:
df = df.copy()

# Convertir la fecha y crear las columnas
df['DIA'] = pd.to_datetime(df['DIA'])
df['Mes'] = df['DIA'].dt.month
df['Año'] = df['DIA'].dt.year

plt.figure(figsize=(12, 6))
sns.boxplot(
    x='Mes', 
    y='Frio (Kw) (Consolidado EE)', 
    data=df, 
    hue='Mes', 
    palette='coolwarm', 
    legend=False
)

plt.title('Distribución del Consumo de Frío por Mes (Comprobación Visual de Estacionalidad)')
plt.xlabel('Mes del Año (1 = Enero, 12 = Diciembre)')
plt.ylabel('Consumo de Frío (Kw)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Generar el boxplot por Año
sns.boxplot(
    x='Año', 
    y='Frio (Kw) (Consolidado EE)', 
    data=df, 
    hue='Año', 
    palette='viridis', # Paleta de colores distinta para que se vea bien
    legend=False
)

# Configuración de etiquetas y título
plt.title('Distribución y Comparación Anual del Consumo de Frío', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Año', fontsize=12)
plt.ylabel('Consumo de Frío (Kw)', fontsize=12)

# Formato visual
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Mostrar el gráfico
plt.show()

In [ ]:
df['Dia_Num'] = df['DIA'].dt.dayofweek

# Mapeamos los números a los nombres de los días
nombres_dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df['Dia_Semana'] = df['Dia_Num'].map(nombres_dias)

# Definimos el orden lógico para que el gráfico empiece el Lunes y termine el Domingo
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

plt.figure(figsize=(10, 6))

sns.boxplot(
    x='Dia_Semana', 
    y='Frio (Kw) (Consolidado EE)', 
    data=df, 
    order=orden_dias, # Obligamos a que respete el orden de Lunes a Domingo
    hue='Dia_Semana',
    palette='Set2', 
    legend=False
)

plt.title('Distribución del Consumo de Frío por Día de la Semana', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Día de la Semana', fontsize=12)
plt.ylabel('Consumo de Frío (Kw)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
df['Semana_Año'] = df['DIA'].dt.isocalendar().week

plt.figure(figsize=(16, 6))
sns.boxplot(
    x='Semana_Año', 
    y='Frio (Kw) (Consolidado EE)', 
    data=df, 
    color='skyblue'
)

plt.title('Consumo de Frío por Semana del Año (Estacionalidad Detallada)', fontsize=14, fontweight='bold')
plt.xlabel('Semana del Año (1 a 52)', fontsize=12)
plt.ylabel('Consumo de Frío (Kw)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Mostramos solo semanas pares en el eje X para que no se amontone el texto
plt.xticks(ticks=range(1, 53, 2), labels=range(2, 54, 2))
plt.tight_layout()
plt.show()

In [ ]:
df_temp = df.set_index('DIA')

# Agrupamos por semana ('W') y calculamos el promedio de consumo de esa semana
df_semanal = df_temp['Frio (Kw) (Consolidado EE)'].resample('W').mean()

plt.figure(figsize=(15, 6))
plt.plot(df_semanal.index, df_semanal.values, color='#ff7f0e', linewidth=2)

plt.title('Evolución Histórica Promediada por Semana (Tendencia Limpia)', fontsize=14, fontweight='bold')
plt.xlabel('Fecha (Semanas)', fontsize=12)
plt.ylabel('Promedio Semanal de Frío (Kw)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
df_numerico = df.select_dtypes(include=['float64', 'int64'])

# 2. Calculamos la matriz de correlación de TODO, pero nos quedamos solo con la columna del Target
# Usamos .abs() para considerar las relaciones negativas y positivas más fuertes por igual
correlaciones_target = df_numerico.corr()['Target_Frio_Manana'].abs().sort_values(ascending=False)

# 3. Obtenemos el nombre de las 15 variables más correlacionadas
# Usamos head(16) porque la primera variable será 'Target_Frio_Manana' (correlación perfecta de 1.0)
top_15_variables = correlaciones_target.head(16).index

# Mostrar en consola cuáles son esas variables y su correlación real (con su signo original)
print("Las 15 variables más correlacionadas con el Target son:")
print(df_numerico.corr()['Target_Frio_Manana'][top_15_variables])

# 4. Filtramos el DataFrame original para quedarnos SOLO con estas 16 variables (Target + 15 features)
df_top_15 = df_numerico[top_15_variables]

# 5. Calculamos la matriz de correlación (16x16)
matriz_top_15 = df_top_15.corr()

# 6. Graficamos el Heatmap
plt.figure(figsize=(14, 12))

# annot_kws={"size": 9} achica un poco el tamaño de los números para que entren bien
sns.heatmap(
    matriz_top_15, 
    annot=True, 
    fmt=".2f", 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5,
    annot_kws={"size": 9}
)

plt.title('Heatmap: Top 15 Variables más correlacionadas con Target_Frio_Manana', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
predictores = top_15_variables.drop('Target_Frio_Manana')
X = df_top_15[predictores].dropna() # VIF y las matrices fallan si hay valores nulos (NaN)

print("--- 1. PARES CON ALTA CORRELACIÓN (> 0.80) ---")
# Calculamos la matriz de correlación solo entre los predictores
matriz_predictores = X.corr().abs()

# Extraemos el triángulo superior de la matriz para no tener duplicados
upper_tri = matriz_predictores.where(np.triu(np.ones(matriz_predictores.shape), k=1).astype(bool))

# Buscamos pares con correlación mayor a 0.80 (umbral de alerta de multicolinealidad)
pares_altos = [
    (columna, indice, upper_tri[columna][indice]) 
    for columna in upper_tri.columns 
    for indice in upper_tri.index 
    if upper_tri[columna][indice] > 0.80
]

if pares_altos:
    for col, idx, val in pares_altos:
        print(f"Alerta: '{col}' y '{idx}' tienen una correlación de {val:.2f}")
else:
    print("No se encontraron pares con correlación > 0.80")

print("\n--- 2. FACTOR DE INFLACIÓN DE LA VARIANZA (VIF) ---")
# Para que statsmodels calcule el VIF correctamente, debemos agregar una constante (intercepto)
X_con_constante = sm.add_constant(X)

# Calculamos el VIF para cada variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_con_constante.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_con_constante.values, i) 
    for i in range(X_con_constante.shape[1])
]

# Filtramos la constante para mostrar solo nuestras variables y ordenamos
vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values(by='VIF', ascending=False)
print(vif_data.to_string(index=False))

In [ ]:
target = df['Target_Frio_Manana'].dropna()

# Crear un lienzo con 2 subgráficos (1 fila, 2 columnas)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ---------------------------------------------------------
# 1. HISTOGRAMA CON CURVA DE DENSIDAD (KDE)
# ---------------------------------------------------------
sns.histplot(target, kde=True, color='#1f77b4', ax=ax1, bins=30)
ax1.set_title('Histograma de Target_Frio_Manana', fontsize=14, fontweight='bold')
ax1.set_xlabel('Consumo de Frío (Kw) [Target]', fontsize=12)
ax1.set_ylabel('Frecuencia (Cantidad de Días)', fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# ---------------------------------------------------------
# 2. QQ-PLOT (Gráfico Cuantil-Cuantil)
# ---------------------------------------------------------
# stats.probplot compara nuestra variable contra una distribución normal teórica ("norm")
stats.probplot(target, dist="norm", plot=ax2)

# Personalización de colores del QQ-Plot para que coincida con el estilo
ax2.get_lines()[0].set_markerfacecolor('#1f77b4') # Puntos azules
ax2.get_lines()[0].set_markeredgecolor('#1f77b4') 
ax2.get_lines()[1].set_color('red')               # Línea de referencia roja
ax2.get_lines()[1].set_linewidth(2)

ax2.set_title('QQ-Plot (Prueba Visual de Normalidad)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Cuantiles Teóricos (Distribución Normal)', fontsize=12)
ax2.set_ylabel('Cuantiles de los Datos Reales', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)

# Ajustar márgenes y mostrar
plt.tight_layout()
plt.show()

In [ ]:
columnas_pairplot = [
    'Target_Frio_Manana',
    'Frio (Kw) (Consolidado EE)', 
    'Servicios (Kw) (Consolidado EE)', 
    'Agua Planta (Hl) (Consolidado Agua)', 
    'Planta de agua (Hl) (Consolidado Agua)', 
    'Meta ET Linea 4 (Consolidado KPI)'
]

# 2. Filtramos el DataFrame original sin nulos
df_pairplot = df[columnas_pairplot].dropna()

# 3. CREAMOS UN DICCIONARIO CON NOMBRES MÁS CORTOS
nombres_cortos = {
    'Target_Frio_Manana': 'Target',
    'Frio (Kw) (Consolidado EE)': 'Frío (Kw)', 
    'Servicios (Kw) (Consolidado EE)': 'Servicios (Kw)', 
    'Agua Planta (Hl) (Consolidado Agua)': 'Agua P. (Hl)', 
    'Planta de agua (Hl) (Consolidado Agua)': 'P. Agua (Hl)', 
    'Meta ET Linea 4 (Consolidado KPI)': 'Meta ET L4'
}

# 4. Renombramos las columnas en nuestro DataFrame temporal
df_pairplot = df_pairplot.rename(columns=nombres_cortos)

# 5. Generamos el Pairplot
g = sns.pairplot(
    df_pairplot, 
    diag_kind='kde', 
    corner=True,
    plot_kws={'alpha': 0.4, 's': 20, 'edgecolor': 'none'}
)

g.fig.suptitle('Pairplot: Target vs Top 5 Predictores', y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
palabras_clave = 'Planta|Produccion|Bodega|Cocina|Elaboracion'


columnas_encontradas = df.columns[df.columns.str.contains(palabras_clave, case=False, na=False)].tolist()

print(f"Se encontraron {len(columnas_encontradas)} columnas:")
for col in columnas_encontradas:
    print("-", col)

palabras_clave_completas = 'DIA|Target_Frio_Manana|Planta|Produccion|Bodega|Cocina|Elaboracion'

df_areas_y_target = df.filter(regex=f'(?i){palabras_clave_completas}')

In [ ]:
cols_ee = [
    'EE Planta / Hl (Consolidado KPI)',
    'EE Elaboracion / Hl (Consolidado KPI)',
    'EE Bodega / Hl (Consolidado KPI)',
    'EE Cocina / Hl (Consolidado KPI)',
    'EE Resto Planta / Hl (Consolidado KPI)'
]

cols_agua = [
    'Agua Planta / Hl (Consolidado KPI)',
    'Agua Bodega / Hl (Consolidado KPI)',
    'Agua Cocina / Hl (Consolidado KPI)',
    'Agua Planta de Agua/Hl (Consolidado KPI)',
    'Produccion Agua / Hl (Consolidado KPI)'
]

cols_et = [
    'ET Planta / Hl (Consolidado KPI)',
    'ET Bodega/Hl (Consolidado KPI)',
    'ET Cocina/Hl (Consolidado KPI)'
]

cols_aire = [
    'Aire Planta / Hl (Consolidado KPI)',
    'Aire Elaboracion / Hl (Consolidado KPI)',
    'Aire Cocina / Hl (Consolidado KPI)',
    'Aire Bodega / Hl (Consolidado KPI)'
]

# 2. Creamos un lienzo grande con 4 subgráficos (2 filas y 2 columnas)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3. Agrupamos la información para iterar y graficar fácilmente
categorias = [
    ('Energía Eléctrica (EE)', cols_ee, axes[0, 0], 'YlOrBr'), # Amarillo/Naranja
    ('Consumo de Agua', cols_agua, axes[0, 1], 'Blues'),       # Azul
    ('Energía Térmica (ET)', cols_et, axes[1, 0], 'Reds'),     # Rojo
    ('Aire Comprimido', cols_aire, axes[1, 1], 'Greens')       # Verde
]

# 4. Generamos cada uno de los 4 gráficos
for titulo, columnas, ax, paleta in categorias:
    
    # Graficar usando solo las columnas correspondientes
    sns.boxplot(data=df[columnas], ax=ax, palette=paleta)
    
    # Configuración de títulos
    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=10)
    ax.set_ylabel('Consumo / Hl', fontsize=11)
    
    # Magia para acortar los nombres largos del Eje X: 
    # Nos quedamos solo con la parte antes de la barra "/"
    nombres_cortos = [col.split(' /')[0].split('/')[0] for col in columnas]
    
    ax.set_xticks(range(len(columnas)))
    ax.set_xticklabels(nombres_cortos, rotation=35, ha='right', fontsize=10)
    
    # Grilla de fondo para facilitar la lectura
    ax.grid(axis='y', linestyle='--', alpha=0.6)

# 5. Ajustar márgenes para que nada se superponga y mostrar
plt.tight_layout()
plt.show()